# Study Rubin shear matching

Analyzes the Rubin metadetect shear match: SNR distributions, catalog coverage, and spatial distributions of good vs bad matches.

In [ ]:
# Standard imports
import matplotlib.pyplot as plt
import numpy as np
import tables_io
import hpmcm

## Configuration

In [ ]:
tract = 8006

In [ ]:
# Load Rubin shear match output tables and input catalog files
shear_rubin_in = tables_io.read(f'{tract}/shear_object_{tract}.parq')
shear_rubin_ns = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_ns.parq')
shear_rubin_1m = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_1m.parq')
shear_rubin_1p = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_1p.parq')
shear_rubin_2m = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_2m.parq')
shear_rubin_2p = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_2p.parq')    
shear_rubin_obj_stats = tables_io.read(f'{tract}/rubin_md_match_object_stats.pq')
shear_rubin_obj_assoc = tables_io.read(f'{tract}/rubin_md_match_object_assoc.pq')
input_files = [
    f'{tract}/shear_object_reduced_uncleaned_{tract}_ns.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1p.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2p.parq',
]    
catalog_ids = [0, 1, 2, 3, 4]

In [ ]:
# Build wide joined table
joined_shear_obj = hpmcm.output_tables.buildJoinedObjectTable(
    shear_rubin_obj_stats,
    shear_rubin_obj_assoc,
    input_files,
    catalog_ids,
)

In [ ]:
# SNR histograms by catalog detection bits
bins = np.logspace(0, 5, 51)
for i in range(5):
    mask_bit = 1 << i
    mask = np.bitwise_and(joined_shear_obj.catalog_mask, mask_bit) == mask_bit
    _ = plt.hist(joined_shear_obj[mask].snr, bins=bins, alpha=0.1)
_ = plt.xscale('log')
_ = plt.yscale('log')

In [ ]:
_ = plt.hist2d(joined_shear_obj.catalog_mask, joined_shear_obj.snr, bins=(np.linspace(0.5, 31.5, 32), np.logspace(0, 5, 51)), norm='log')
_ = plt.yscale('log')

In [ ]:
# Objects detected in all 5 Rubin catalogs have catalog_mask == 31
good_mask = joined_shear_obj.catalog_mask == 31
bad_mask = joined_shear_obj.catalog_mask != 31
good_sub = joined_shear_obj[good_mask]
bad_sub = joined_shear_obj[bad_mask]


In [ ]:
_ = plt.scatter(good_sub.x_cell_coadd_1[::5], good_sub.y_cell_coadd_1[::5], s=1)
_ = plt.scatter(bad_sub.x_cell_coadd_1[::5], bad_sub.y_cell_coadd_1[::5], s=1)

In [ ]:
_ = plt.scatter(good_sub.x_cell_coadd_2[::5], good_sub.y_cell_coadd_2[::5], s=1)
_ = plt.scatter(bad_sub.x_cell_coadd_2[::5], bad_sub.y_cell_coadd_2[::5], s=1)

In [ ]:
_ = plt.scatter(good_sub.x_pix[::5], good_sub.y_pix[::5], s=1)
_ = plt.scatter(bad_sub.x_pix[::5], bad_sub.y_pix[::5], s=1)

In [ ]:
_ = plt.scatter(good_sub.ra[::5], good_sub.dec[::5], s=1)
_ = plt.scatter(bad_sub.ra[::5], bad_sub.dec[::5], s=1)